# 05-3. 트리의 앙상블

여러 결정 트리의 예측을 결합하는 랜덤 포레스트와 엑스트라 트리, 트리를 순서대로 추가하는 그레이디언트 부스팅 계열을 비교한다.

## 학습 목표

- 앙상블이 여러 트리를 결합하는 이유 이해
- 랜덤 포레스트와 엑스트라 트리의 차이 구분
- OOB 점수와 특성 중요도 해석
- 그레이디언트 부스팅의 순차 학습 이해
- 히스토그램 기반 부스팅과 외부 라이브러리 결과 비교

## 1. 데이터 준비

5-1, 5-2와 같은 와인 데이터를 사용한다. 입력 특성은 `alcohol`, `sugar`, `pH`이고 타깃은 `class`다.

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# 5장에서 계속 사용한 와인 데이터를 불러온다.
wine = pd.read_csv('https://bit.ly/wine_csv_data')

# 입력 특성과 정답 타깃을 분리한다.
data = wine[['alcohol', 'sugar', 'pH']]
target = wine['class']

# 전체 데이터의 20%를 최종 테스트 세트로 분리한다.
train_input, test_input, train_target, test_target = train_test_split(
    data,
    target,
    test_size=0.2,
    random_state=42
)

## 2. 랜덤 포레스트

랜덤 포레스트는 서로 조금씩 다른 결정 트리를 여러 개 학습하고 분류 결과를 합친다.

각 트리는 다음 두 가지 무작위성을 가진다.

1. 훈련 샘플을 중복 허용으로 뽑는 부트스트랩 샘플
2. 노드마다 전체 특성 중 일부만 후보로 사용

트리마다 다른 데이터와 특성을 보게 만들어 한 트리의 과대적합을 줄인다.

In [3]:
from sklearn.model_selection import cross_validate
from sklearn.ensemble import RandomForestClassifier

# 여러 결정 트리를 학습해 예측을 합치는 랜덤 포레스트를 만든다.
# n_jobs=-1은 가능한 CPU 코어를 모두 사용한다.
rf = RandomForestClassifier(
    n_jobs=-1,
    random_state=42
)

# 5-폴드 교차 검증으로 훈련 점수와 검증 점수를 함께 확인한다.
scores = cross_validate(
    rf,
    train_input,
    train_target,
    return_train_score=True,
    n_jobs=-1
)

print(np.mean(scores['train_score']), np.mean(scores['test_score']))

0.9973541965122431 0.8905151032797809


In [4]:
# 전체 훈련 세트로 랜덤 포레스트를 다시 학습한다.
rf.fit(train_input, train_target)

# 특성 중요도는 alcohol, sugar, pH 순서로 출력된다.
print(rf.feature_importances_)

[0.23167441 0.50039841 0.26792718]


In [5]:
# OOB 점수를 계산하도록 랜덤 포레스트를 다시 만든다.
# 각 트리의 부트스트랩 샘플에 뽑히지 않은 데이터를 검증에 사용한다.
rf = RandomForestClassifier(
    oob_score=True,
    n_jobs=-1,
    random_state=42
)
rf.fit(train_input, train_target)

# 별도 검증 세트 없이 얻은 OOB 정확도를 확인한다.
print(rf.oob_score_)

0.8934000384837406


### 실습 결과

- 평균 훈련 정확도: `0.9974`
- 평균 검증 정확도: `0.8905`
- OOB 정확도: `0.8934`

교차 검증의 평균 검증 점수와 OOB 점수가 비슷하다. OOB 샘플은 해당 트리의 부트스트랩 샘플에 선택되지 않은 데이터이므로, 별도 검증 세트처럼 사용할 수 있다.

### 특성 중요도

| 특성 | 중요도 |
|---|---:|
| `alcohol` | `0.2317` |
| `sugar` | `0.5004` |
| `pH` | `0.2679` |

5-1의 단일 결정 트리보다 중요도가 세 특성에 더 고르게 분산됐다. 여러 트리가 서로 다른 특성 후보를 사용한 결과다.

## 3. 엑스트라 트리

엑스트라 트리도 여러 결정 트리의 예측을 합치지만 랜덤 포레스트보다 노드 분할에 더 큰 무작위성을 준다.

In [11]:
from sklearn.ensemble import ExtraTreesClassifier

# 엑스트라 트리는 전체 훈련 샘플을 사용하고,
# 각 노드의 분할 기준을 랜덤하게 선택해 트리 사이의 다양성을 높인다.
et = ExtraTreesClassifier(
    n_jobs=-1,
    random_state=42
)

# 5-폴드 교차 검증으로 훈련·검증 평균 정확도를 확인한다.
scores = cross_validate(
    et,
    train_input,
    train_target,
    return_train_score=True,
    n_jobs=-1
)

print(np.mean(scores['train_score']), np.mean(scores['test_score']))

0.9974503966084433 0.8887848893166506


In [12]:
# 전체 훈련 세트로 엑스트라 트리를 학습한다.
et.fit(train_input, train_target)

# alcohol, sugar, pH의 특성 중요도를 확인한다.
print(et.feature_importances_)

[0.20183568 0.52242907 0.27573525]


### 실습 결과

- 평균 훈련 정확도: `0.9975`
- 평균 검증 정확도: `0.8888`

랜덤 포레스트의 `0.8905`와 비슷하지만 이 실습에서는 조금 낮다.

### 특성 중요도

| 특성 | 중요도 |
|---|---:|
| `alcohol` | `0.2018` |
| `sugar` | `0.5224` |
| `pH` | `0.2757` |

> **이전 질문과 연결 — “랜덤 포레스트와 엑스트라 트리는 무엇이 다른가?”**  
> 둘 다 여러 트리를 합치는 앙상블이다. 랜덤 포레스트는 부트스트랩 샘플을 사용하고 후보 특성 안에서 좋은 분할을 찾는다. 엑스트라 트리는 기본적으로 전체 훈련 샘플을 사용하며 분할 기준도 무작위로 정해 트리 사이의 차이를 더 크게 만든다.

> **이전 질문과 연결 — “노드를 무작위로 분할한다는 것은 무슨 뜻인가?”**  
> 특성과 임계값을 모두 최적으로 고르는 대신, 무작위로 뽑은 분할 후보를 사용한다는 뜻이다. 클래스를 아무렇게나 나누는 것이 아니라 선택 과정에 무작위성을 넣는 것이다.

> **이전 질문과 연결 — “결정 트리의 `splitter='random'`이 곧 엑스트라 트리인가?”**  
> 아니다. `splitter='random'`은 한 개의 결정 트리에 무작위 분할을 적용한 설정이다. `ExtraTreesClassifier`는 이런 무작위성이 큰 트리를 여러 개 학습해 결과를 합치는 앙상블이다.

## 4. 그레이디언트 부스팅

랜덤 포레스트와 엑스트라 트리는 여러 트리를 비교적 독립적으로 만든다. 그레이디언트 부스팅은 트리를 순서대로 추가하며 앞선 모델이 부족했던 부분을 다음 트리가 보완한다.

In [14]:
from sklearn.ensemble import GradientBoostingClassifier

# 얕은 결정 트리를 순서대로 추가하며 앞선 모델의 오차를 보완한다.
gb = GradientBoostingClassifier(random_state=42)

# 기본 설정의 훈련·검증 평균 정확도를 확인한다.
scores = cross_validate(
    gb,
    train_input,
    train_target,
    return_train_score=True,
    n_jobs=-1
)

print(np.mean(scores['train_score']), np.mean(scores['test_score']))

0.8881086892152563 0.8720430147331015


In [15]:
# 트리 수를 500개로 늘리고 각 트리의 반영 정도를 0.2로 설정한다.
gb = GradientBoostingClassifier(
    n_estimators=500,
    learning_rate=0.2,
    random_state=42
)

# 모델 복잡도를 높인 뒤 훈련·검증 평균 정확도를 다시 확인한다.
scores = cross_validate(
    gb,
    train_input,
    train_target,
    return_train_score=True,
    n_jobs=-1
)

print(np.mean(scores['train_score']), np.mean(scores['test_score']))

0.9464595437171814 0.8780082549788999


In [17]:
# 전체 훈련 세트로 그레이디언트 부스팅을 학습한다.
gb.fit(train_input, train_target)

# alcohol, sugar, pH의 특성 중요도를 확인한다.
print(gb.feature_importances_)

[0.15887763 0.6799705  0.16115187]


### 실습 결과

| 설정 | 평균 훈련 정확도 | 평균 검증 정확도 |
|---|---:|---:|
| 기본 설정 | `0.8881` | `0.8720` |
| `n_estimators=500`, `learning_rate=0.2` | `0.9465` | `0.8780` |

트리 수와 학습률을 높인 설정은 훈련 점수와 검증 점수를 모두 높였다. 다만 훈련·검증 점수 차이도 커졌으므로, 트리 수를 늘리면 항상 좋은 것은 아니다.

- `n_estimators`: 순서대로 추가할 트리 수
- `learning_rate`: 각 트리의 보정 결과를 반영하는 정도

### 특성 중요도

| 특성 | 중요도 |
|---|---:|
| `alcohol` | `0.1589` |
| `sugar` | `0.6800` |
| `pH` | `0.1612` |

## 5. 히스토그램 기반 그레이디언트 부스팅

연속형 특성을 여러 구간으로 묶은 뒤 분할을 탐색해 일반적인 그레이디언트 부스팅보다 빠르게 학습한다.

In [19]:
from sklearn.ensemble import HistGradientBoostingClassifier

# 연속형 특성을 구간으로 나누어 빠르게 학습하는
# 히스토그램 기반 그레이디언트 부스팅을 만든다.
hgb = HistGradientBoostingClassifier(random_state=42)

# 5-폴드 교차 검증으로 훈련·검증 평균 정확도를 확인한다.
scores = cross_validate(
    hgb,
    train_input,
    train_target,
    return_train_score=True
)

print(np.mean(scores['train_score']), np.mean(scores['test_score']))

0.9321723946453317 0.8801241948619236


In [20]:
from sklearn.inspection import permutation_importance

# 전체 훈련 세트로 히스토그램 기반 모델을 학습한다.
hgb.fit(train_input, train_target)

# 각 특성을 무작위로 섞었을 때 정확도가 얼마나 감소하는지 측정한다.
# n_repeats=10은 각 특성의 섞기를 10번 반복해 평균을 구한다.
result = permutation_importance(
    hgb,
    train_input,
    train_target,
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

# alcohol, sugar, pH 순서의 평균 중요도를 확인한다.
print(result.importances_mean)

[0.08876275 0.23438522 0.08027708]


In [21]:
# 테스트 세트에서도 같은 방식으로 순열 중요도를 계산한다.
# 훈련 세트뿐 아니라 새로운 데이터에서도 중요한 특성인지 확인할 수 있다.
result = permutation_importance(
    hgb,
    test_input,
    test_target,
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

print(result.importances_mean)

[0.05969231 0.20238462 0.049     ]


In [22]:
# 모든 모델 선택과 분석이 끝난 뒤
# 히스토그램 기반 모델의 최종 테스트 정확도를 확인한다.
hgb.score(test_input, test_target)

0.8723076923076923

### 실습 결과

- 평균 훈련 정확도: `0.9322`
- 평균 검증 정확도: `0.8801`
- 최종 테스트 정확도: `0.8723`

`cross_validate()`의 `test_score`는 각 폴드의 검증 점수이고, `hgb.score(test_input, test_target)`만 최종 테스트 세트 점수다.

### 순열 중요도

순열 중요도는 한 특성의 값을 무작위로 섞은 뒤 모델 성능이 얼마나 떨어지는지 측정한다. 많이 떨어질수록 모델이 그 특성에 크게 의존한다.

| 특성 | 훈련 세트 | 테스트 세트 |
|---|---:|---:|
| `alcohol` | `0.0888` | `0.0597` |
| `sugar` | `0.2344` | `0.2024` |
| `pH` | `0.0803` | `0.0490` |

훈련 세트와 테스트 세트 모두 `sugar`를 섞었을 때 정확도가 가장 크게 감소했다.

## 6. XGBoost와 LightGBM

두 모델도 결정 트리를 순서대로 추가하는 부스팅 계열 구현이다. 원본 실습에서는 기본 설정으로 교차 검증 결과를 비교한다.

In [23]:
from xgboost import XGBClassifier

# XGBoost의 히스토그램 기반 트리 학습 방식을 사용한다.
xgb = XGBClassifier(
    tree_method='hist',
    random_state=42
)

# 5-폴드 교차 검증으로 훈련·검증 평균 정확도를 확인한다.
scores = cross_validate(
    xgb,
    train_input,
    train_target,
    return_train_score=True,
    n_jobs=-1
)

print(np.mean(scores['train_score']), np.mean(scores['test_score']))

0.9572351925731766 0.8781988968682904


In [24]:
from lightgbm import LGBMClassifier

# LightGBM 분류 모델을 기본 설정으로 만든다.
lgb = LGBMClassifier(random_state=42)

# 5-폴드 교차 검증으로 훈련·검증 평균 정확도를 확인한다.
scores = cross_validate(
    lgb,
    train_input,
    train_target,
    return_train_score=True,
    n_jobs=-1
)

print(np.mean(scores['train_score']), np.mean(scores['test_score']))

/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


0.935828414851749 0.8801251203079884


### 실습 결과 비교

| 모델 | 평균 훈련 정확도 | 평균 검증 정확도 |
|---|---:|---:|
| 랜덤 포레스트 | `0.9974` | `0.8905` |
| 엑스트라 트리 | `0.9975` | `0.8888` |
| 그레이디언트 부스팅 기본 | `0.8881` | `0.8720` |
| 그레이디언트 부스팅 조정 | `0.9465` | `0.8780` |
| 히스토그램 그레이디언트 부스팅 | `0.9322` | `0.8801` |
| XGBoost | `0.9572` | `0.8782` |
| LightGBM | `0.9358` | `0.8801` |

이 실습의 평균 검증 정확도는 랜덤 포레스트가 가장 높다. 다만 모든 모델을 같은 수준으로 튜닝한 비교는 아니므로, 이 결과만으로 항상 랜덤 포레스트가 우수하다고 일반화할 수는 없다.

## 정리

```text
랜덤 포레스트
→ 부트스트랩 샘플과 일부 특성으로 서로 다른 트리 생성
→ 여러 트리의 예측을 합침

엑스트라 트리
→ 전체 훈련 샘플 사용
→ 분할 기준에 더 큰 무작위성 부여
→ 여러 트리의 예측을 합침

그레이디언트 부스팅
→ 트리를 순서대로 추가
→ 앞선 모델의 부족한 부분을 다음 트리가 보완

히스토그램 기반 부스팅
→ 특성값을 구간으로 묶어 빠르게 분할 탐색
```

핵심은 여러 트리를 단순히 많이 만드는 것이 아니라, **트리마다 다른 데이터·특성·분할을 보게 하거나 이전 트리의 오차를 다음 트리가 보완하게 만드는 것**이다.